# Introduction

This notebook presents an implementaiton of the **inference pipeline for a GraphRAG via Semantic Search**, using:
- Tech Stack: Neo4j and LangChain
- Query Method: Vector Search/Semantic Similarity Search and Enriched Search via Cypher query language

**Setting up**

You will need to have a running Neo4j instance, which there are some ways to do that. To replicate the results here, you will need:
1. create a [free Neo4j database instance in their Aura cloud service](https://neo4j.com/cloud/platform/aura-graph-database/) and
2. import the [FDA Adverse Event Reporting System (FAERS)](https://open.fda.gov/data/faers/) database from the healthcare-analytics guide that contains information on adverse event and medication error reports submitted to [FDA](https://www.fda.gov/).
*This approach makes the inference pipeline independent of the indexing pipeline. So, you want skip the graph creation and jump directly to the RAG part, you can start with this notebook*.

**Sources**
1. [DeepLearning.AI | Knowledge Graphs for RAG](https://learn.deeplearning.ai/courses/knowledge-graphs-rag/lesson/8/chatting-with-the-knowledge-graph)
2. [LangChain | Neo4j](https://python.langchain.com/v0.2/docs/integrations/graphs/neo4j_cypher/)
3. [LangChain | Chains - RetrievalQAWithSourcesChain](https://python.langchain.com/v0.1/docs/modules/chains/)

In [1]:
! pip install --quiet \
              langchain==0.2.14 \
              langchain-openai==0.1.20 \
              langchain-community==0.2.10 \
              neo4j-graphrag==1.2.0 \
              python-dotenv==1.0.1

In [19]:
import os
import textwrap

from dotenv import load_dotenv, find_dotenv
from langchain.chains import RetrievalQA
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [3]:
use_google_drive_api = True
google_drive_path = 'gdrive/My Drive/Colab Notebooks/Toolkit-NLP/GraphRAG_with_Neo4j_and_LangChain'

if(use_google_drive_api):
  from google.colab import drive
  drive.mount('/content/gdrive/', force_remount=True)
  # Changing the current working directory to the notebook folder
  os.chdir(google_drive_path)
  print(os.getcwd())

Mounted at /content/gdrive/
/content/gdrive/My Drive/Colab Notebooks/Toolkit-NLP/GraphRAG_with_Neo4j_and_LangChain


In [4]:
# Enverionment variables
_ = load_dotenv(find_dotenv(), override=True) # read local .env file

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

# 1. Import Graph

In [11]:
kg = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)

In [12]:
kg.refresh_schema()
print(kg.schema)

Node properties:
AgeGroup {id: INTEGER, ageGroup: STRING}
Case {id: INTEGER, age: FLOAT, ageUnit: STRING, eventDate: DATE_TIME, gender: STRING, primaryid: INTEGER, reportDate: DATE_TIME, reporterOccupation: STRING}
Drug {id: INTEGER, name: STRING, primarySubstabce: STRING}
Manufacturer {id: INTEGER, manufacturerName: STRING}
Outcome {id: INTEGER, code: STRING, outcome: STRING}
Reaction {id: INTEGER, description: STRING}
ReportSource {id: INTEGER, name: STRING, code: STRING}
Therapy {id: INTEGER, primaryid: INTEGER}
Relationship properties:
IS_CONCOMITANT {indication: STRING, doseAmount: STRING, doseUnit: STRING, drugSequence: INTEGER, route: STRING}
IS_PRIMARY_SUSPECT {indication: STRING, doseAmount: FLOAT, doseUnit: STRING, drugSequence: BOOLEAN, route: STRING}
IS_SECONDARY_SUSPECT {indication: STRING, doseAmount: FLOAT, doseUnit: STRING, drugSequence: INTEGER, route: STRING}
PRESCRIBED {drugSequence: INTEGER, endYear: INTEGER, startYear: INTEGER}
The relationships:
(:Case)-[:FALLS_UN

In [14]:
query = """
MATCH (c:Case)-[h:HAS_REACTION]->(r:Reaction)
RETURN c,h,r
LIMIT 1
"""

kg.query(query)

[{'c': {'ageUnit': 'YR',
   'gender': 'F',
   'reportDate': neo4j.time.DateTime(2021, 8, 27, 0, 0, 0, 0, tzinfo=<UTC>),
   'reporterOccupation': 'Physician',
   'id': 0,
   'primaryid': 100654764,
   'age': 56.0,
   'eventDate': neo4j.time.DateTime(2014, 3, 25, 0, 0, 0, 0, tzinfo=<UTC>)},
  'h': ({'ageUnit': 'YR',
    'gender': 'F',
    'reportDate': neo4j.time.DateTime(2021, 8, 27, 0, 0, 0, 0, tzinfo=<UTC>),
    'reporterOccupation': 'Physician',
    'id': 0,
    'primaryid': 100654764,
    'age': 56.0,
    'eventDate': neo4j.time.DateTime(2014, 3, 25, 0, 0, 0, 0, tzinfo=<UTC>)},
   'HAS_REACTION',
   {'description': 'Pulmonary embolism', 'id': 4454}),
  'r': {'description': 'Pulmonary embolism', 'id': 4454}}]

# 2. Create and Populate the Vector Index

In [15]:
# Global constants
OPENAI_ENDPOINT = 'https://api.openai.com/v1/embeddings'
VECTOR_INDEX_NAME = "vector_index"
VECTOR_NODE_LABEL = 'Reaction'
VECTOR_SOURCE_PROPERTY = 'description'
VECTOR_EMBEDDING_PROPERTY = 'embedding'

In [16]:
# Create vector index
kg.query("""
         CREATE VECTOR INDEX `vector_index` IF NOT EXISTS
          FOR (r:Reaction) ON (r.embedding)
          OPTIONS { indexConfig: {
            `vector.dimensions`: 1536,
            `vector.similarity_function`: 'cosine'
         }}
""")

[]

In [17]:
kg.query("SHOW INDEXES")

[{'id': 7,
  'name': 'ageUnit_Case',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Case'],
  'properties': ['ageUnit'],
  'indexProvider': 'range-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 6,
  'name': 'age_Case',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Case'],
  'properties': ['age'],
  'indexProvider': 'range-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 20,
  'name': 'code_Outcome',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Outcome'],
  'properties': ['code'],
  'indexProvider': 'range-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 26,
  'name': 'code_ReportSource',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'lab

In [18]:
kg.query("""
    MATCH (r:Reaction) WHERE r.embedding IS NULL
    WITH r, genai.vector.encode(
      r.description,
      "OpenAI",
      {
        token: $openAiApiKey,
        endpoint: $openAiEndpoint
      }) AS vector
    CALL db.create.setNodeVectorProperty(r, "embedding", vector)
    """,
    params={"openAiApiKey":OPENAI_API_KEY, "openAiEndpoint": OPENAI_ENDPOINT} )

[]

In [20]:
kg.refresh_schema()
print(kg.schema)

Node properties:
AgeGroup {id: INTEGER, ageGroup: STRING}
Case {id: INTEGER, age: FLOAT, ageUnit: STRING, eventDate: DATE_TIME, gender: STRING, primaryid: INTEGER, reportDate: DATE_TIME, reporterOccupation: STRING}
Drug {id: INTEGER, name: STRING, primarySubstabce: STRING}
Manufacturer {id: INTEGER, manufacturerName: STRING}
Outcome {id: INTEGER, code: STRING, outcome: STRING}
Reaction {id: INTEGER, description: STRING, embedding: LIST}
ReportSource {id: INTEGER, name: STRING, code: STRING}
Therapy {id: INTEGER, primaryid: INTEGER}
Relationship properties:
IS_CONCOMITANT {indication: STRING, doseAmount: STRING, doseUnit: STRING, drugSequence: INTEGER, route: STRING}
IS_INTERACTING {indication: STRING, doseAmount: INTEGER, doseUnit: STRING, drugSequence: INTEGER, route: STRING}
IS_PRIMARY_SUSPECT {indication: STRING, doseAmount: FLOAT, doseUnit: STRING, drugSequence: BOOLEAN, route: STRING}
IS_SECONDARY_SUSPECT {indication: STRING, doseAmount: FLOAT, doseUnit: STRING, drugSequence: INTE

In [48]:
# ## 2.1 Add source property to use RAG Chain
#
# create_source_property_to_reaction_node_query = """
# MATCH (r:Reaction)
# SET r.source = r.id
# RETURN count(r) as numberOfReactions
# """
#
# kg.query(create_source_property_to_reaction_node_query)

# 3. Perform Vector Search

In [55]:
def neo4j_vector_search(question, top_k=3):
    """Search for similar nodes using the Neo4j vector index"""
    vector_search_query = """
      WITH genai.vector.encode(
        $question,
        "OpenAI",
        {
          token: $openAiApiKey,
          endpoint: $openAiEndpoint
        }) AS question_embedding
      CALL db.index.vector.queryNodes($index_name, $top_k, question_embedding) yield node, score
      RETURN node.description AS text, score, node {.id} AS metadata
    """
    similar = kg.query(vector_search_query,
                      params={
                        'question': question,
                        'openAiApiKey':OPENAI_API_KEY,
                        'openAiEndpoint': OPENAI_ENDPOINT,
                        'index_name':VECTOR_INDEX_NAME,
                        'top_k': top_k})
    return similar

In [56]:
question = 'How many cases have there been in which the drug has been ineffective?'

search_results = neo4j_vector_search(question)

search_results

[{'text': 'Drug ineffective',
  'score': 0.9281768798828125,
  'metadata': {'id': 4469}},
 {'text': 'Therapeutic product ineffective',
  'score': 0.925506591796875,
  'metadata': {'id': 6095}},
 {'text': 'Drug ineffective for unapproved indication',
  'score': 0.9183197021484375,
  'metadata': {'id': 5068}}]

### Customize the results of the similarity search using Cypher

In [57]:
def neo4j_vector_search_enriched(question, top_k=3):
    """Search for similar nodes using the Neo4j vector index and expand context using a window"""
    vector_search_query = """
      WITH genai.vector.encode(
        $question,
        "OpenAI",
        {
          token: $openAiApiKey,
          endpoint: $openAiEndpoint
        }) AS question_embedding
      CALL db.index.vector.queryNodes($index_name, $top_k, question_embedding) yield node, score
      MATCH (c:Case)-[h:HAS_REACTION]->(node)
      WITH collect(toString(c.eventDate)) as caseList, node, score
      RETURN "Reaction: " + node.description + "\nCase Dates: " + apoc.text.join(caseList, ", ") as text,
          score,
          node {.id} AS metadata
    """
    similar = kg.query(vector_search_query,
                      params={
                        'question': question,
                        'openAiApiKey':OPENAI_API_KEY,
                        'openAiEndpoint': OPENAI_ENDPOINT,
                        'index_name':VECTOR_INDEX_NAME,
                        'top_k': top_k})
    return similar

In [58]:
question = 'How many cases have there been in which the drug has been ineffective?'

search_results = neo4j_vector_search_enriched(question)

search_results

[{'text': 'Reaction: Drug ineffective\nCase Dates: 2014-09-05T00:00:00Z, 2019-03-11T00:00:00Z, 2019-03-20T00:00:00Z, 2017-04-15T00:00:00Z, 2018-02-04T00:00:00Z, 2018-03-30T00:00:00Z, 2010-05-06T00:00:00Z, 2018-12-05T00:00:00Z, 2021-05-01T00:00:00Z, 2018-04-01T00:00:00Z, 2018-12-31T00:00:00Z, 2019-02-13T00:00:00Z, 2019-01-11T00:00:00Z, 2019-02-27T00:00:00Z, 2020-03-23T00:00:00Z, 2019-03-17T00:00:00Z, 2021-03-22T00:00:00Z, 2021-02-19T00:00:00Z, 2019-07-11T00:00:00Z, 2019-06-18T00:00:00Z, 2013-11-06T00:00:00Z, 2019-08-23T00:00:00Z, 2019-08-01T00:00:00Z, 2019-11-09T00:00:00Z, 2019-12-03T00:00:00Z, 2019-12-14T00:00:00Z, 2019-12-18T00:00:00Z, 2017-04-01T00:00:00Z, 2018-04-19T00:00:00Z, 2019-09-09T00:00:00Z, 2019-05-11T00:00:00Z, 2019-10-18T00:00:00Z, 2020-05-18T00:00:00Z, 2020-05-21T00:00:00Z, 2018-08-17T00:00:00Z, 2021-08-22T00:00:00Z, 2019-04-30T00:00:00Z, 2020-06-12T00:00:00Z, 2020-06-29T00:00:00Z, 2016-03-24T00:00:00Z, 2021-01-28T00:00:00Z, 2020-03-24T00:00:00Z, 2019-09-26T00:00:00Z, 202

# 4. Chating with the graph

In [59]:
# Set up vector store and retriever

neo4j_vector_store = Neo4jVector.from_existing_graph(
    embedding=OpenAIEmbeddings(),
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name=VECTOR_INDEX_NAME,
    node_label=VECTOR_NODE_LABEL,
    text_node_properties=[VECTOR_SOURCE_PROPERTY],
    embedding_node_property=VECTOR_EMBEDDING_PROPERTY,
)

retriever = neo4j_vector_store.as_retriever(search_kwargs={'k':3})

In [60]:
# Instantiate an LLM

llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    temperature=0.4,
    model_name="gpt-4o-mini",
)

In [61]:
# Instantiate an QA Chain

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
)

def generate_response_and_display_with_sources(question: str) -> str:
    """Pretty print the chain's response to a question"""
    response = chain.invoke({"query": question}, return_only_outputs=False)
    print(textwrap.fill(response['result'], 80))

In [62]:
question = 'How many cases have there been in which the drug has been ineffective?'

generate_response_and_display_with_sources(question)

I don't know.


### Customize the results of the similarity search using Cypher

In [63]:
retrieval_query_enriched = """
MATCH (c:Case)-[h:HAS_REACTION]->(node)
WITH collect(toString(c.eventDate)) as caseList, node, score
RETURN "Reaction: " + node.description + "\nCase Dates: " + apoc.text.join(caseList, ", ") as text,
    score,
    node {.id} AS metadata
"""

In [64]:
# Set up a QA chain that will use the window retrieval query

vector_store_enriched = Neo4jVector.from_existing_index(
    embedding=OpenAIEmbeddings(),
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="neo4j",
    index_name=VECTOR_INDEX_NAME,
    text_node_property=VECTOR_SOURCE_PROPERTY,
    retrieval_query=retrieval_query_enriched, # NEW!!!
)

# Create a retriever from the vector store
retriever_enriched = vector_store_enriched.as_retriever(search_kwargs={'k':3})

# Create a chatbot Question & Answer chain from the retriever
chain_enriched = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever_enriched
)

def generate_response_enriched_and_display_with_sources(question: str) -> str:
    """Pretty print the chain's response to a question"""
    response = chain_enriched.invoke({"query": question}, return_only_outputs=False)
    print(textwrap.fill(response['result'], 80))

In [65]:
question = 'How many cases have there been in which the drug has been ineffective?'

generate_response_enriched_and_display_with_sources(question)

There have been a total of 12 cases in which the drug has been reported as
ineffective. This includes cases of "Drug ineffective" (10 cases) and "Drug
ineffective for unapproved indication" (6 cases). However, since some cases may
overlap, the total number of unique cases may be less than the sum of these two
categories.
